In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(".env", override=True)
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"


In [8]:
def get_completion_from_messages(
    messages,
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_tokens=500
):
    system_instruction = ""
    contents = []

    for message in messages:
        if message["role"] == "system":
            system_instruction = message["content"]

        elif message["role"] == "user":
            contents.append({
                "role": "user",
                "parts": [{"text": message["content"]}]
            })

        elif message["role"] == "assistant":
            contents.append({
                "role": "model",
                "parts": [{"text": message["content"]}]
            })

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    return response.text

In [9]:
delimiter = "####"

system_message = f"""
You will be provided with customer service queries.
The customer service query will be delimited with
{delimiter} characters.

Classify each query into a primary category
and a secondary category.

Provide your output in JSON format with the
keys: primary and secondary.

Primary categories: Billing, Technical Support,
Account Management, or General Inquiry.

Billing secondary categories:
Unsubscribe or upgrade
Add a payment method
Explanation for charge
Dispute a charge

Technical Support secondary categories:
General troubleshooting
Device compatibility
Software updates

Account Management secondary categories:
Password reset
Update personal information
Close account
Account security

General Inquiry secondary categories:
Product information
Pricing
Feedback
Speak to a human
"""

user_message = "I want you to delete my profile and all of my user data"

messages = [
    {
        "role": "system",
        "content": system_message
    },
    {
        "role": "user",
        "content": f"{delimiter}{user_message}{delimiter}"
    }
]

response = get_completion_from_messages(messages)

print(response)

```json
{
  "primary": "Account Management",
  "secondary": "Close account"
}
```


In [10]:
user_message = f"""\
Tell me more about your flat screen tvs"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = get_completion_from_messages(messages)
print(response)

```json
{
  "primary": "General Inquiry",
  "secondary": "Product information"
}
```
